# PANICLE Tutorial: GWAS Analysis of Sorghum Diversity Panel

This notebook demonstrates a complete genome-wide association study (GWAS) workflow using PANICLE on real sorghum data.

**Dataset:**
- 725 sorghum accessions from a diversity panel
- 170,072 genetic markers
- Photosynthesis-related traits (Phi2, PhiNPQ, PhiNO, FvP/FmP)

**What you'll learn:**
1. Loading and aligning genotype/phenotype data
2. Computing population structure (PCs and kinship)
3. Running different GWAS methods (GLM, MLM, FarmCPU, BLINK)
4. Comparing and interpreting results

## Setup

First, let's import the required packages and set up our environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# PANICLE imports
from panicle.pipelines.gwas import GWASPipeline

# For manual/lower-level access (optional)
from panicle.data.loaders import load_genotype_file, load_phenotype_file, match_individuals
from panicle.matrix.pca import PANICLE_PCA
from panicle.matrix.kinship import PANICLE_K_VanRaden
from panicle.association.glm import PANICLE_GLM
from panicle.association.mlm_loco import PANICLE_MLM_LOCO
from panicle.association.farmcpu import PANICLE_FarmCPU
from panicle.association.blink import PANICLE_BLINK

print("PANICLE loaded successfully!")

## 1. Data Loading

We'll use the sorghum diversity panel data included in the repository.

In [ ]:
# Define file paths
GENOTYPE_FILE = "../sorghum_data/SbDiv_RNAseq_GeneticMarkers_Mangal2025.vcf.gz"
PHENOTYPE_FILE = "../sorghum_data/SpATS_genotype_BLUEs_all_traits_fixed.csv"
OUTPUT_DIR = "./tutorial_results"

# Preview the phenotype data
pheno_df = pd.read_csv(PHENOTYPE_FILE)
print(f"Phenotype file: {len(pheno_df)} samples")
print(f"Columns: {list(pheno_df.columns)}")
pheno_df.head()

### Understanding the Traits

These are photosynthesis efficiency traits measured using chlorophyll fluorescence:

- **Phi2**: Quantum yield of photosystem II (photosynthetic efficiency)
- **PhiNPQ**: Non-photochemical quenching (heat dissipation)
- **PhiNO**: Non-regulated energy dissipation
- **FvP/FmP**: Variable to maximum fluorescence ratio

In [ ]:
# Look at trait distributions
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
traits = ['BLUE_Phi2_corrected', 'BLUE_PhiNPQ_corrected', 
          'BLUE_PhiNO_corrected', 'BLUE_FvP_over_FmP_corrected']

for ax, trait in zip(axes.flat, traits):
    pheno_df[trait].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_xlabel(trait.replace('BLUE_', '').replace('_corrected', ''))
    ax.set_ylabel('Count')

plt.suptitle('Distribution of Photosynthesis Traits', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Using GWASPipeline (High-Level API)

The easiest way to run GWAS is using the `GWASPipeline` class, which handles all the steps automatically.

In [ ]:
# Initialize the pipeline
pipeline = GWASPipeline(output_dir=OUTPUT_DIR)

# Load data
print("Loading data...")
start = time.time()
pipeline.load_data(
    phenotype_file=PHENOTYPE_FILE,
    genotype_file=GENOTYPE_FILE,
    trait_columns=['BLUE_Phi2_corrected']  # Start with one trait
)
print(f"Data loaded in {time.time() - start:.2f} seconds")

In [ ]:
# Align samples between genotype and phenotype
pipeline.align_samples()

print(f"\nAligned dataset:")
print(f"  Samples: {pipeline.genotype_matrix.n_individuals}")
print(f"  Markers: {pipeline.genotype_matrix.n_markers}")

In [ ]:
# Compute population structure
print("Computing population structure...")
start = time.time()
pipeline.compute_population_structure(
    n_pcs=3,                # Number of principal components
    calculate_kinship=True  # Useful for MLM without map data and for kinship visualization
)
print(f"Population structure computed in {time.time() - start:.2f} seconds")

### Visualize Population Structure

In [ ]:
# Plot first two PCs
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PC1 vs PC2
axes[0].scatter(pipeline.pcs[:, 0], pipeline.pcs[:, 1], alpha=0.5, s=10)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('Population Structure (PC1 vs PC2)')

# Kinship heatmap (subsample for visualization)
n_show = min(100, pipeline.kinship.shape[0])
im = axes[1].imshow(pipeline.kinship[:n_show, :n_show], cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[1].set_title(f'Kinship Matrix (first {n_show} samples)')
plt.colorbar(im, ax=axes[1], label='Relatedness')

plt.tight_layout()
plt.show()

## 3. Run GWAS Analysis

Now let's run GWAS using multiple methods to compare their results.

In [ ]:
# Run GWAS with multiple methods
print("Running GWAS analysis...")
print("This may take a few minutes.\n")

start = time.time()
pipeline.run_analysis(
    traits=['BLUE_Phi2_corrected'],
    methods=['GLM', 'MLM', 'FarmCPU', 'BLINK'],
    max_iterations=10,  # For FarmCPU and BLINK
    ncpus=1,
    parallel_mode='off',  # Keep the tutorial portable across notebook/script execution
    outputs=['all_marker_pvalues', 'significant_marker_pvalues', 'manhattan', 'qq']
)
print(f"\nTotal analysis time: {time.time() - start:.2f} seconds")

## 4. Examine Results

Let's load and explore the results.

In [ ]:
# Load the full results
results = pd.read_csv(f"{OUTPUT_DIR}/GWAS_BLUE_Phi2_corrected_all_results.csv")
print(f"Results shape: {results.shape}")
print(f"\nColumns: {list(results.columns)}")
results.head()

In [ ]:
# Calculate significance threshold (Bonferroni)
n_markers = len(results)
bonferroni = 0.05 / n_markers
print(f"Bonferroni threshold (0.05/{n_markers:,}): {bonferroni:.2e}")

# Count significant hits per method
methods = ['GLM', 'MLM', 'FarmCPU', 'BLINK']
print("\nSignificant hits per method:")
for method in methods:
    p_col = f"{method}_P"
    if p_col in results.columns:
        n_sig = (results[p_col] < bonferroni).sum()
        print(f"  {method}: {n_sig}")

In [ ]:
# Top 10 markers by MLM p-value
if 'MLM_P' in results.columns:
    top_markers = results.nsmallest(10, 'MLM_P')[['MARKER', 'CHROM', 'POS', 'MAF', 'GLM_P', 'MLM_P']]
    print("Top 10 markers by MLM p-value:")
    display(top_markers)
else:
    top_markers = results.nsmallest(10, 'GLM_P')[['MARKER', 'CHROM', 'POS', 'MAF', 'GLM_P']]
    print("Top 10 markers by GLM p-value:")
    display(top_markers)

### Compare Methods

In [ ]:
# Compare p-values between methods
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# GLM vs MLM
if 'MLM_P' in results.columns:
    ax = axes[0]
    x = -np.log10(results['GLM_P'].clip(lower=1e-300))
    y = -np.log10(results['MLM_P'].clip(lower=1e-300))
    ax.scatter(x, y, alpha=0.1, s=1)
    ax.plot([0, max(x.max(), y.max())], [0, max(x.max(), y.max())], 'r--', lw=1)
    ax.set_xlabel('-log10(GLM P)')
    ax.set_ylabel('-log10(MLM P)')
    ax.set_title('GLM vs MLM')

# GLM vs FarmCPU
if 'FarmCPU_P' in results.columns:
    ax = axes[1]
    x = -np.log10(results['GLM_P'].clip(lower=1e-300))
    y = -np.log10(results['FarmCPU_P'].clip(lower=1e-300))
    ax.scatter(x, y, alpha=0.1, s=1)
    ax.plot([0, max(x.max(), y.max())], [0, max(x.max(), y.max())], 'r--', lw=1)
    ax.set_xlabel('-log10(GLM P)')
    ax.set_ylabel('-log10(FarmCPU P)')
    ax.set_title('GLM vs FarmCPU')

# MLM vs BLINK
if 'MLM_P' in results.columns and 'BLINK_P' in results.columns:
    ax = axes[2]
    x = -np.log10(results['MLM_P'].clip(lower=1e-300))
    y = -np.log10(results['BLINK_P'].clip(lower=1e-300))
    ax.scatter(x, y, alpha=0.1, s=1)
    ax.plot([0, max(x.max(), y.max())], [0, max(x.max(), y.max())], 'r--', lw=1)
    ax.set_xlabel('-log10(MLM P)')
    ax.set_ylabel('-log10(BLINK P)')
    ax.set_title('MLM vs BLINK')

plt.tight_layout()
plt.show()

### Custom Manhattan Plot

In [ ]:
def plot_manhattan(results, p_col, title, ax=None):
    """Create a Manhattan plot."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 4))
    
    # Get unique chromosomes and assign colors
    chroms = results['CHROM'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(chroms)))
    chrom_colors = {c: colors[i % len(colors)] for i, c in enumerate(sorted(chroms))}
    
    # Calculate cumulative positions
    results = results.copy()
    results['color'] = results['CHROM'].map(chrom_colors)
    results['-log10p'] = -np.log10(results[p_col].clip(lower=1e-300))
    
    # Sort by chromosome and position
    results = results.sort_values(['CHROM', 'POS'])
    
    # Plot
    for chrom in sorted(chroms):
        subset = results[results['CHROM'] == chrom]
        ax.scatter(range(len(subset)), subset['-log10p'], 
                   c=[chrom_colors[chrom]], s=2, alpha=0.5, label=chrom)
    
    # Significance line
    bonf = -np.log10(0.05 / len(results))
    ax.axhline(bonf, color='red', linestyle='--', linewidth=1, label=f'Bonferroni (p={0.05/len(results):.1e})')
    
    ax.set_xlabel('Genomic Position')
    ax.set_ylabel('-log10(P-value)')
    ax.set_title(title)
    
    return ax

# Plot Manhattan for MLM
if 'MLM_P' in results.columns:
    fig, ax = plt.subplots(figsize=(14, 4))
    plot_manhattan(results, 'MLM_P', 'Manhattan Plot - MLM (Phi2)', ax)
    plt.tight_layout()
    plt.show()

## 5. Low-Level API Example

For more control, you can use the individual functions directly.

In [ ]:
# Load data manually
print("Loading data with low-level API...")

# Load genotype
genotype, sample_ids, map_data = load_genotype_file(GENOTYPE_FILE)
print(f"Genotype: {genotype.n_individuals} samples x {genotype.n_markers} markers")

# Load phenotype
phenotype_df = load_phenotype_file(PHENOTYPE_FILE)
print(f"Phenotype: {len(phenotype_df)} samples")

# Align samples
aligned_pheno, _, sample_indices, summary = match_individuals(
    phenotype_df=phenotype_df,
    individual_ids=sample_ids
)
geno_data = genotype._data[sample_indices, :]
print(f"Aligned: {summary['n_common']} common samples")

In [ ]:
# Prepare phenotype matrix (required format: n x 2 with index and values)
trait_values = aligned_pheno['BLUE_Phi2_corrected'].values.astype(np.float64)
phe_matrix = np.column_stack([np.arange(len(trait_values)), trait_values])

# Compute PCs
pcs = PANICLE_PCA(geno_data, pcs_keep=3, verbose=False)
print(f"PCs shape: {pcs.shape}")

# Run GLM directly
print("\nRunning GLM...")
start = time.time()
glm_results = PANICLE_GLM(phe=phe_matrix, geno=geno_data, CV=pcs, verbose=False)
print(f"GLM completed in {time.time() - start:.3f} seconds")
print(f"Min p-value: {np.min(glm_results.pvalues):.2e}")

## 6. Summary

In this tutorial, we:

1. **Loaded real sorghum data** - 725 samples with 170K markers
2. **Computed population structure** - PCs and kinship matrix
3. **Ran multiple GWAS methods** - GLM, MLM, FarmCPU, BLINK
4. **Compared results** - Visualized method agreement
5. **Demonstrated both APIs** - High-level pipeline and low-level functions

### Key Takeaways

- **GLM** is fastest but doesn't account for population structure
- **MLM** controls for relatedness using kinship matrix
- **FarmCPU** and **BLINK** use iterative approaches for improved power
- Use the **GWASPipeline** for most analyses; use low-level functions for custom workflows

### Next Steps

- Try analyzing other traits in the dataset
- Experiment with different numbers of PCs
- Use MLM with map data to enable LOCO + exact top-hit refinement
- Check the [API Reference](api_reference.md) for all available options

In [ ]:
# Print output files
import os
print("Generated output files:")
if os.path.exists(OUTPUT_DIR):
    for f in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
        print(f"  {f} ({size:.1f} KB)")